# Phase B.2.3 — Causal gender steering on Gemma 2 9B (layers 20 and 31)

**Prerequisite:** B.2.2 (`exp_gemma9b_probes.ipynb`) PASSED — `gemma9b_probes_layer20_31.json` in the directory.

**Goal:** Test whether the features identified by the probes at L20 and L31 *control* the model's behavior (not only correlate). Replicates the 2B causal protocol with the expanded 228-item benchmark.

**Setup:**
- Target features: top-1 PT-discriminative for gender at L20 and L31 (from `gemma9b_probes_layer20_31.json`).
- Since B.2.2 confirmed there are no "generic" features on 9B (top-1 distinct per phenomenon), we use top-1 directly without a fallback test.
- Controls: 5 norm-matched features per layer.
- Benchmark: 228 items (40 adjectives × 3 templates × 2 genders, identical to 2B).
- Fluency: 10 open-ended generations (5 seeds × 2 prompts) per feature.

**Outputs:**
- `gemma9b_steering_layer20_31.json` — baseline, ablated, controls, Fisher p per layer.

**Estimated runtime:** ~40-50 min on Colab L4.

## 1. Installation and environment

In [ ]:
!pip install -q sae-lens transformer-lens scipy

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} ({total_gb:.1f} GB)")
    assert total_gb >= 20, "Precisa de >=20GB VRAM. Use L4 ou A100."
else:
    raise RuntimeError("Precisa de GPU.")

## 2. Configuration

In [ ]:
MODEL_NAME = "gemma-2-9b"
HF_MODEL_ID = f"google/{MODEL_NAME}"
LAYERS = [20, 31]
WIDTH = "16k"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"
DTYPE = torch.bfloat16

N_CONTROLS = 5
SEEDS = [42, 123, 456, 789, 1337]
MAX_NEW_TOKENS = 30
TEMPERATURE = 0.7

STEERING_PROMPTS = [
    "A menina bonita foi à",
    "A professora dedicada explicou",
]

PROBES_JSON_PATH = "gemma9b_probes_layer20_31.json"
OUT_JSON = "gemma9b_steering_layer20_31.json"

GENDER_ERROR_PATTERNS = [
    r"\bA\s+\w+\s+(bonito|alto|magro|gordo|baixo|jovem|velho|novo|inteligente|cansado|feliz|triste|bravo|calmo|nervoso|preocupado|animado|esperto|dedicado|aplicado|talentoso|famoso|simpático|antipático)\b",
    r"\bO\s+\w+\s+(bonita|alta|magra|gorda|baixa|jovem|velha|nova|inteligente|cansada|feliz|triste|brava|calma|nervosa|preocupada|animada|esperta|dedicada|aplicada|talentosa|famosa|simpática|antipática)\b",
    r"\bele\s+(estava|ficou|era|parecia|se sentiu)\s+(bonita|alta|magra|gorda|cansada|dedicada)\b",
    r"\bela\s+(estava|ficou|era|parecia|se sentiu)\s+(bonito|alto|magro|gordo|cansado|dedicado)\b",
]

print(f"Config: layers={LAYERS}, seeds={len(SEEDS)}, prompts={len(STEERING_PROMPTS)}, controls={N_CONTROLS}")

## 3. Load model and SAEs

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.cuda.empty_cache()
gc.collect()

print(f"Loading {HF_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=DTYPE,
    device_map={"": device},
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

n_model_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"layers: {n_model_layers} | d_model: {d_model}")
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
from sae_lens import SAE

saes = {}
for L in LAYERS:
    sae_id = f"layer_{L}/width_{WIDTH}/canonical"
    print(f"Loading SAE {sae_id}...")
    s = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=device)
    if isinstance(s, tuple):
        s = s[0]
    s = s.to(DTYPE)
    saes[L] = s
    assert s.cfg.d_in == d_model

D_SAE = saes[LAYERS[0]].cfg.d_sae
print(f"VRAM em uso: {torch.cuda.memory_allocated()/1024**3:.2f} GB | d_sae={D_SAE}")

## 4. Carregar features candidatas do B.2.2

In [ ]:
import json

with open(PROBES_JSON_PATH) as f:
    probes_data = json.load(f)

TARGETS = {}
for L in LAYERS:
    top1 = probes_data["probe_results"]["gender"][str(L)]["top_features"][0]
    TARGETS[L] = {
        "feature_id": top1["feature_id"],
        "diff": top1["diff"],
        "effect_size": top1["effect_size"],
        "mean_pos": top1["mean_pos"],
        "mean_neg": top1["mean_neg"],
    }
    print(f"L{L} target: feat {TARGETS[L]['feature_id']} | diff={TARGETS[L]['diff']:+.3f} | d={TARGETS[L]['effect_size']:+.3f}")
    print(f"        mean_pos (F)={TARGETS[L]['mean_pos']:.3f} | mean_neg (M)={TARGETS[L]['mean_neg']:.3f}")

## 5. Ablation hook

Applies feature ablation: passes the residual stream through the SAE encoder, zeros the target feature, decodes, and preserves the reconstruction error (i.e., the part of the residual stream the SAE does not capture remains intact).

In [ ]:
from contextlib import contextmanager

def make_ablation_hook(sae, feature_id):
    fid = int(feature_id)
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            h, rest = output[0], output[1:]
        else:
            h, rest = output, None
        with torch.no_grad():
            feat = sae.encode(h)
            modified = feat.clone()
            modified[..., fid] = 0.0
            recon_orig = sae.decode(feat)
            recon_mod = sae.decode(modified)
            new_h = h - recon_orig + recon_mod
        if rest is not None:
            return (new_h,) + rest
        return new_h
    return hook

@contextmanager
def ablate(layer, feature_id):
    """Context manager to install/remove the ablation hook."""
    if feature_id is None:
        yield
        return
    hook = make_ablation_hook(saes[layer], feature_id)
    handle = model.model.layers[layer].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()

test_text = "A menina bonita foi à"
test_ids = tokenizer(test_text, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logits_b = model(test_ids).logits
with ablate(LAYERS[0], TARGETS[LAYERS[0]]["feature_id"]):
    with torch.no_grad():
        logits_a = model(test_ids).logits
diff = (logits_b.float() - logits_a.float()).abs().max().item()
print(f"Sanity: ablation modifica logits (max |Δ|={diff:.4f})")

## 6. Gender benchmark (228 items — identical to 2B)

In [ ]:
import random

ADJECTIVES = [
    ("bonito", "bonita"), ("alto", "alta"), ("baixo", "baixa"),
    ("gordo", "gorda"), ("magro", "magra"), ("velho", "velha"),
    ("novo", "nova"), ("feio", "feia"), ("rico", "rica"),
    ("pobre", "pobre"), ("cansado", "cansada"), ("animado", "animada"),
    ("dedicado", "dedicada"), ("organizado", "organizada"),
    ("preocupado", "preocupada"), ("calmo", "calma"),
    ("nervoso", "nervosa"), ("famoso", "famosa"),
    ("talentoso", "talentosa"), ("corajoso", "corajosa"),
    ("preguiçoso", "preguiçosa"), ("cuidadoso", "cuidadosa"),
    ("carinhoso", "carinhosa"), ("generoso", "generosa"),
    ("sério", "séria"), ("simpático", "simpática"),
    ("antipático", "antipática"), ("tímido", "tímida"),
    ("atrevido", "atrevida"), ("educado", "educada"),
    ("esperto", "esperta"), ("lento", "lenta"),
    ("rápido", "rápida"), ("quieto", "quieta"),
    ("barulhento", "barulhenta"), ("chato", "chata"),
    ("divertido", "divertida"), ("salgado", "salgada"),
    ("amargo", "amarga"), ("doce", "doce"),
]

SUBJECTS_FEM = [
    "A menina", "A professora", "A médica", "A engenheira", "A advogada",
    "A diretora", "A aluna", "A cantora", "A cozinheira", "A vizinha",
    "A mãe", "A filha", "A irmã", "A avó", "A tia",
    "A enfermeira", "A motorista", "A secretária", "A escritora", "A atriz",
]

SUBJECTS_MASC = [
    "O menino", "O professor", "O médico", "O engenheiro", "O advogado",
    "O diretor", "O aluno", "O cantor", "O cozinheiro", "O vizinho",
    "O pai", "O filho", "O irmão", "O avô", "O tio",
    "O enfermeiro", "O motorista", "O secretário", "O escritor", "O ator",
]

TEMPLATES = [
    "{subj} é muito",
    "{subj} parece",
    "{subj} ficou muito",
]

random.seed(42)

GENDER_BENCHMARK = []
for adj_m, adj_f in ADJECTIVES:
    if adj_m == adj_f:
        continue
    for template in TEMPLATES:
        subj_f = random.choice(SUBJECTS_FEM)
        subj_m = random.choice(SUBJECTS_MASC)
        GENDER_BENCHMARK.append((template.format(subj=subj_f), adj_f, adj_m, "F"))
        GENDER_BENCHMARK.append((template.format(subj=subj_m), adj_m, adj_f, "M"))

random.shuffle(GENDER_BENCHMARK)
n_fem = sum(1 for _, _, _, g in GENDER_BENCHMARK if g == "F")
n_masc = sum(1 for _, _, _, g in GENDER_BENCHMARK if g == "M")
print(f"Benchmark total: {len(GENDER_BENCHMARK)} items ({n_fem} F + {n_masc} M)")

## 7. Evaluation and generation functions

In [ ]:
import re
from tqdm.auto import tqdm

def get_token_prob(probs, word):
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    ids_no_space = tokenizer.encode(word, add_special_tokens=False)
    all_ids = list(set(ids + ids_no_space))
    return sum(probs[tid].item() for tid in all_ids)

def evaluate_benchmark(benchmark, layer=None, feature_id=None, desc=""):
    results = []
    with ablate(layer, feature_id):
        for ctx, correct_adj, incorrect_adj, gender in tqdm(benchmark, desc=desc, leave=False):
            input_ids = tokenizer.encode(ctx, return_tensors="pt").to(device)
            with torch.no_grad():
                logits = model(input_ids).logits
            probs = torch.softmax(logits[0, -1, :].float(), dim=-1)
            p_correct = get_token_prob(probs, correct_adj)
            p_incorrect = get_token_prob(probs, incorrect_adj)
            results.append({
                "gender": gender,
                "p_correct": p_correct,
                "p_incorrect": p_incorrect,
                "is_correct": p_correct > p_incorrect,
            })
    return results

def compute_stats(results):
    fem = [r for r in results if r["gender"] == "F"]
    masc = [r for r in results if r["gender"] == "M"]
    n_correct = sum(1 for r in results if r["is_correct"])
    fem_correct = sum(1 for r in fem if r["is_correct"])
    masc_correct = sum(1 for r in masc if r["is_correct"])
    return {
        "overall": n_correct / len(results),
        "feminine": fem_correct / max(len(fem), 1),
        "masculine": masc_correct / max(len(masc), 1),
        "n_total": len(results),
        "n_fem": len(fem), "n_masc": len(masc),
        "fem_correct": fem_correct, "masc_correct": masc_correct,
    }

def generate_text(prompt, seed, layer=None, feature_id=None):
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    with ablate(layer, feature_id):
        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def detect_gender_errors(text):
    matches = []
    for pattern in GENDER_ERROR_PATTERNS:
        matches.extend(re.findall(pattern, text))
    return matches

print("Functions loaded.")

## 8. Select norm-matched control features per layer

In [ ]:
CONTROLS = {}
for L in LAYERS:
    target_fid = TARGETS[L]["feature_id"]
    sae = saes[L]
    target_norm = sae.W_dec.data[target_fid].float().norm().item()
    all_norms = sae.W_dec.data.float().norm(dim=1)
    norm_diffs = (all_norms - target_norm).abs()
    norm_diffs[target_fid] = float("inf")
    ctrl_ids = norm_diffs.argsort()[:N_CONTROLS].tolist()
    CONTROLS[L] = {
        "target_norm": target_norm,
        "feature_ids": ctrl_ids,
        "norms": [all_norms[i].item() for i in ctrl_ids],
    }
    print(f"L{L} target {target_fid} (norm={target_norm:.4f})")
    print(f"     controls: {ctrl_ids}")
    print(f"     norms:     {[round(n, 4) for n in CONTROLS[L]['norms']]}")

## 9. Open-ended generations (fluency) — target + controls

In [ ]:
FLUENCY = {}

for L in LAYERS:
    print(f"\n=== Layer {L} ===")
    target_fid = TARGETS[L]["feature_id"]

    print(f"[target {target_fid}] generations with ablation...")
    target_gens = []
    target_errors = 0
    for prompt in STEERING_PROMPTS:
        for seed in SEEDS:
            base = generate_text(prompt, seed)
            ablt = generate_text(prompt, seed, layer=L, feature_id=target_fid)
            errs = detect_gender_errors(ablt)
            target_gens.append({
                "prompt": prompt, "seed": seed,
                "baseline": base, "ablated": ablt,
                "regex_hit": len(errs) > 0, "regex_matches": errs,
            })
            if errs:
                target_errors += 1
    print(f"  errors (regex) in ablation: {target_errors}/{len(target_gens)}")

    print(f"[controls] 1 short generation per (control, seed) for fluency...")
    ctrl_errors_total = 0
    ctrl_n_total = 0
    for ctrl_fid in CONTROLS[L]["feature_ids"]:
        for prompt in STEERING_PROMPTS[:1]:
            for seed in SEEDS[:3]:
                txt = generate_text(prompt, seed, layer=L, feature_id=ctrl_fid)
                errs = detect_gender_errors(txt)
                if errs:
                    ctrl_errors_total += 1
                ctrl_n_total += 1
    print(f"  errors (regex) in controls: {ctrl_errors_total}/{ctrl_n_total}")

    FLUENCY[L] = {
        "target_feature": target_fid,
        "target_generations": target_gens,
        "target_error_count": target_errors,
        "target_n": len(target_gens),
        "control_error_count": ctrl_errors_total,
        "control_n": ctrl_n_total,
    }

## 10. Gender benchmark (228 items × baseline + ablated + 5 controls)

In [ ]:
from scipy.stats import fisher_exact
import numpy as np

BENCH = {}

for L in LAYERS:
    print(f"\n=== Benchmark layer {L} (target feature {TARGETS[L]['feature_id']}) ===")
    target_fid = TARGETS[L]["feature_id"]

    print("  baseline (no ablation)...")
    baseline_res = evaluate_benchmark(GENDER_BENCHMARK, desc=f"L{L} baseline")
    base_stats = compute_stats(baseline_res)

    print("  target ablation...")
    ablated_res = evaluate_benchmark(GENDER_BENCHMARK, layer=L, feature_id=target_fid, desc=f"L{L} ablated")
    abl_stats = compute_stats(ablated_res)

    ctrl_stats_list = []
    for i, ctrl_fid in enumerate(CONTROLS[L]["feature_ids"]):
        res = evaluate_benchmark(GENDER_BENCHMARK, layer=L, feature_id=ctrl_fid, desc=f"L{L} ctrl{i+1}")
        ctrl_stats_list.append(compute_stats(res))

    fem_table = [
        [abl_stats["fem_correct"], base_stats["n_fem"] - abl_stats["fem_correct"]],
        [base_stats["fem_correct"], base_stats["n_fem"] - base_stats["fem_correct"]],
    ]
    masc_table = [
        [abl_stats["masc_correct"], base_stats["n_masc"] - abl_stats["masc_correct"]],
        [base_stats["masc_correct"], base_stats["n_masc"] - base_stats["masc_correct"]],
    ]
    _, fisher_p_fem = fisher_exact(fem_table)
    _, fisher_p_masc = fisher_exact(masc_table)

    fem_drop_pp = 100 * (base_stats["feminine"] - abl_stats["feminine"])
    masc_drop_pp = 100 * (base_stats["masculine"] - abl_stats["masculine"])
    ctrl_overall = [c["overall"] for c in ctrl_stats_list]

    BENCH[L] = {
        "effective_feature": target_fid,
        "baseline": base_stats,
        "ablated": abl_stats,
        "control_accs": ctrl_overall,
        "control_mean_acc": float(np.mean(ctrl_overall)),
        "control_stats": ctrl_stats_list,
        "fisher_p_feminine": float(fisher_p_fem),
        "fisher_p_masculine": float(fisher_p_masc),
        "fem_drop_pp": float(fem_drop_pp),
        "masc_drop_pp": float(masc_drop_pp),
    }

    print(f"  baseline  overall={base_stats['overall']:.4f}  F={base_stats['feminine']:.4f}  M={base_stats['masculine']:.4f}")
    print(f"  ablated   overall={abl_stats['overall']:.4f}  F={abl_stats['feminine']:.4f}  M={abl_stats['masculine']:.4f}")
    print(f"  Δfem={fem_drop_pp:+.2f}pp  Δmasc={masc_drop_pp:+.2f}pp  Fisher p_fem={fisher_p_fem:.4f}  p_masc={fisher_p_masc:.4f}")
    print(f"  control mean acc={float(np.mean(ctrl_overall)):.4f} (individual: {[round(a,4) for a in ctrl_overall]})")

## 11. Cross-layer summary

In [ ]:
print("\n=== Cross-layer summary (9B) ===")
print(f"  {'layer':<7}{'feat':>6}  {'baseline':>10}{'ablated':>10}{'Δfem':>9}{'Δmasc':>9}{'pF':>10}{'ctrl_mean':>11}  fluency_err")
for L in LAYERS:
    b = BENCH[L]
    flu = FLUENCY[L]
    print(f"  L{L:<6}{b['effective_feature']:>6}  "
          f"{b['baseline']['overall']:>10.4f}{b['ablated']['overall']:>10.4f}"
          f"{b['fem_drop_pp']:>+9.2f}{b['masc_drop_pp']:>+9.2f}"
          f"{b['fisher_p_feminine']:>10.4f}{b['control_mean_acc']:>11.4f}"
          f"  {flu['target_error_count']}/{flu['target_n']} (ctrl {flu['control_error_count']}/{flu['control_n']})")

## 12. Cross-scale: 2B vs 9B (causal gender steering)

In [ ]:
REFS_2B = {
    "L9":  {"feature": 10569, "depth_pct": 34.6, "fem_drop_pp":  1.75, "fisher_p_fem": 0.7964, "ctrl_mean_acc": 0.8421},
    "L13": {"feature":  1215, "depth_pct": 50.0, "fem_drop_pp":  0.00, "fisher_p_fem": 1.0000, "ctrl_mean_acc": 0.8421},
    "L17": {"feature":  6011, "depth_pct": 65.4, "fem_drop_pp": 20.18, "fisher_p_fem": 0.0000, "ctrl_mean_acc": 0.8421},
}

print("== 2B (reference, from the paper) ==")
print(f"  {'layer':<6}{'feat':>6}{'depth%':>9}{'Δfem':>9}{'pF':>10}{'ctrl_mean':>11}")
for k, v in REFS_2B.items():
    print(f"  {k:<6}{v['feature']:>6}{v['depth_pct']:>9.1f}{v['fem_drop_pp']:>+9.2f}{v['fisher_p_fem']:>10.4f}{v['ctrl_mean_acc']:>11.4f}")

print("\n== 9B (este experimento) ==")
print(f"  {'layer':<6}{'feat':>6}{'depth%':>9}{'Δfem':>9}{'pF':>10}{'ctrl_mean':>11}")
for L in LAYERS:
    b = BENCH[L]
    depth = 100 * L / n_model_layers
    print(f"  L{L:<5}{b['effective_feature']:>6}{depth:>9.1f}{b['fem_drop_pp']:>+9.2f}{b['fisher_p_feminine']:>10.4f}{b['control_mean_acc']:>11.4f}")

## 13. Save results

In [ ]:
result = {
    "experiment": "gemma9b_steering_multilayer",
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "layers": LAYERS,
    "n_model_layers": n_model_layers,
    "d_sae": D_SAE,
    "n_benchmark_items": len(GENDER_BENCHMARK),
    "seeds": SEEDS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "steering_prompts": STEERING_PROMPTS,
    "n_controls": N_CONTROLS,
    "targets": {str(L): TARGETS[L] for L in LAYERS},
    "controls": {str(L): CONTROLS[L] for L in LAYERS},
    "fluency": {str(L): {k: v for k, v in FLUENCY[L].items() if k != "target_generations"} for L in LAYERS},
    "fluency_generations": {str(L): FLUENCY[L]["target_generations"] for L in LAYERS},
    "benchmark_results": {str(L): BENCH[L] for L in LAYERS},
    "refs_2b": REFS_2B,
}

with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Saved to {OUT_JSON}")

## 14. Interpretation and next steps

**Success criteria (per layer):**

1. **Target feature causality:** ablation reduces accuracy in ONE gender (not both), with Fisher p < 0.01 in that gender and virtually unchanged accuracy in the other.
2. **Specificity:** the mean accuracy of the 5 norm-matched controls stays close to baseline (Δ < 2 pp).
3. **Fluency:** ablation produces some regex-hit agreement errors in the 10 open-ended generations; controls produce 0–1.

**Expected result:** with effect sizes much larger than on 2B (probes indicated d between 6 and 20 in magnitude), a causal effect at least comparable to 2B L17 is expected (Δfem = −20 pp, p < 0.0001).

**Output to the paper:**
- Cross-scale table (2B L9/L13/L17 vs 9B L20/L31) with Δfem, Δmasc, p, ctrl_mean.
- Narrative statement: "The same phenomenon-dependent pattern identified on 2B reproduces on 9B; causal control remains possible with single-feature ablation in the upper half of the model (≥50% depth), with more discriminative and more PT-polarized features."

**Remaining tasks:**
- Update the manuscript with the new "Cross-scale generalization: 2B → 9B" section.
- Compile and review.
- Final commit.